In [ ]:
# 🚀 Install PyTorch Nightly with CUDA 12.8 for RTX 5090 Support
# This will install the latest PyTorch with proper RTX 5090 kernel support

import subprocess
import sys

def install_pytorch_nightly():
    """Install PyTorch nightly build with CUDA 12.8 support"""
    
    print("🔄 Installing PyTorch Nightly with CUDA 12.8 support for RTX 5090...")
    print("⚠️  This may take several minutes to download and install.")
    print("=" * 80)
    
    # The command to install PyTorch nightly with CUDA 12.8
    install_command = [
        sys.executable, "-m", "pip", "install", "--pre", 
        "torch", "torchvision", 
        "--index-url", "https://download.pytorch.org/whl/nightly/cu128"
    ]
    
    try:
        # Run the installation
        print("Running command:")
        print(" ".join(install_command))
        print("\n" + "="*50)
        
        result = subprocess.run(
            install_command,
            capture_output=True,
            text=True,
            timeout=600  # 10 minute timeout
        )
        
        # Print output
        if result.stdout:
            print("📥 Installation Output:")
            print(result.stdout)
        
        if result.stderr:
            print("⚠️  Installation Messages:")
            print(result.stderr)
        
        if result.returncode == 0:
            print("\n✅ PyTorch nightly installation completed successfully!")
            print("🔄 Please RESTART your kernel after installation completes")
            print("📝 After restarting, check PyTorch version with: torch.__version__")
        else:
            print(f"\n❌ Installation failed with return code: {result.returncode}")
            
    except subprocess.TimeoutExpired:
        print("⏰ Installation timed out (took longer than 10 minutes)")
        print("💡 Try running the command manually in terminal:")
        print("pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128")
        
    except Exception as e:
        print(f"❌ Installation error: {e}")
        print("💡 Try running manually in terminal:")
        print("pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128")

# Run the installation
install_pytorch_nightly()

print("\n" + "="*80)
print("🎯 NEXT STEPS AFTER INSTALLATION:")
print("1. RESTART your Jupyter kernel (Kernel → Restart)")
print("2. Run: import torch; print(torch.__version__)")
print("3. Test GPU: torch.cuda.is_available()")
print("4. Try loading your model again - RTX 5090 should now work!")
print("="*80)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import re
import os
import glob
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, mean_absolute_error, mean_squared_error
import numpy as np
import warnings
import gc

# Check GPU availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU device:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())
    print("Current GPU:", torch.cuda.current_device())
    print("GPU memory allocated:", torch.cuda.memory_allocated(0) / 1024**3, "GB")
    print("GPU memory reserved:", torch.cuda.memory_reserved(0) / 1024**3, "GB")
else:
    print("⚠️ WARNING: No GPU detected! Running on CPU will be extremely slow.")
    print("Please ensure:")
    print("  1. You have an NVIDIA GPU installed")
    print("  2. CUDA drivers are installed")
    print("  3. PyTorch CUDA version matches your CUDA driver")

# Disable tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Paths
image_path = r"D:\Dementia-Thesis - Don't access without permission\Cookie-Theft-stimulus.png"
transcription_dir = r"D:\Dementia-Thesis - Don't access without permission\cha_par_only"
ground_truth_path = r"D:\Dementia-Thesis - Don't access without permission\2_classes.csv"
output_csv_path = r"D:\Dementia-Thesis - Don't access without permission\qwen_few_shot.csv"

# Clear any existing GPU memory
gc.collect()
torch.cuda.empty_cache()

# Load ground truth
print("Loading ground truth data...")
df_gt = pd.read_csv(ground_truth_path)

# Normalize ground truth labels to lowercase for consistent matching
if 'dx' in df_gt.columns:
    df_gt['dx'] = df_gt['dx'].apply(lambda x: str(x).lower() if pd.notna(x) else None)

print(f"Ground truth loaded: {len(df_gt)} total records")

# Filter to only records WITH valid dx labels
df_gt_valid = df_gt[df_gt['dx'].notna()].copy()
print(f"Records with valid dx labels: {len(df_gt_valid)}")
print(f"Records WITHOUT dx labels (will be skipped): {len(df_gt) - len(df_gt_valid)}")
print(f"Unique dx labels: {df_gt_valid['dx'].unique()}")

# Get all .cha files
cha_files_all = sorted(glob.glob(os.path.join(transcription_dir, "*.cha")))
print(f"\nFound {len(cha_files_all)} total .cha files")

# Filter to only process files with valid dx labels
valid_ids = set(df_gt_valid['id'].astype(str))
cha_files = [f for f in cha_files_all if os.path.splitext(os.path.basename(f))[0] in valid_ids]

skipped_count = len(cha_files_all) - len(cha_files)
print(f"Files to process (with valid dx): {len(cha_files)}")
print(f"Files skipped (no dx label): {skipped_count}")

# Load image (shared across all analyses)
image = Image.open(image_path)

# ============================================================================
# FEW-SHOT EXAMPLES - Curated clinical cases for in-context learning
# ============================================================================
FEW_SHOT_EXAMPLES = """
---
EXAMPLE 1: control (Healthy Control)
TRANSCRIPTION: "In the picture, there's a woman standing at the kitchen sink washing dishes. The water is overflowing from the sink onto the floor. Behind her, two children are trying to steal cookies from the cookie jar on top of the counter. The boy is standing on a stool that's tipping over dangerously, and the girl is reaching up to help him. Outside the window, you can see trees and bushes. The woman doesn't seem to notice what's happening behind her because she's focused on washing the dishes."

ASSESSMENT:
DIAGNOSIS: control
MMSE_SCORE: 28
REASONING: The description is coherent, well-structured, and captures all major elements of the scene (woman, children, cookie theft, tipping stool, overflowing sink). Speech flows naturally with appropriate use of pronouns, spatial relationships, and temporal sequences. No significant word-finding difficulties or semantic confusion observed.

---
EXAMPLE 2: probablead (Alzheimer's Disease - Mild)
TRANSCRIPTION: "There's a lady... she's doing something at the... the thing where you wash. And the... it's coming out... the water. Some kids are there... they want the... you know, the sweet things in the jar. The boy is up high and it's not... not safe. She doesn't see them doing that."

ASSESSMENT:
DIAGNOSIS: probablead
MMSE_SCORE: 20
REASONING: Description shows mild cognitive impairment with word-finding difficulties (circumlocution: "thing where you wash" for sink, "sweet things" for cookies). Core elements are identified but with reduced detail and vocabulary precision. Some grammatical structure remains intact but discourse is fragmented.

---
EXAMPLE 3: probablead (Alzheimer's Disease - Moderate)
TRANSCRIPTION: "The woman... she's there. Water... water is... yes. Children... they're... the boy... and the thing... falls. She... she doesn't... the cookies... or something."

ASSESSMENT:
DIAGNOSIS: probablead
MMSE_SCORE: 14
REASONING: Severe discourse impairment with significant word-finding deficits, incomplete phrases, and loss of narrative coherence. Patient identifies some elements but cannot form complete descriptions or maintain thematic continuity. Marked reduction in information content and grammatical complexity consistent with moderate AD.

---
EXAMPLE 4: control (Healthy Control)
TRANSCRIPTION: "This Cookie Theft picture shows a domestic scene in a kitchen. A mother is washing dishes while water overflows onto the floor, suggesting she's distracted or preoccupied. Meanwhile, her two children are engaged in sneaking cookies from a jar on the counter. The boy has climbed onto a wobbly stool and is reaching for the cookie jar while his sister steadies him or assists. The stool appears unstable, creating a sense of danger. Through the window, there's a view of the yard with vegetation visible."

ASSESSMENT:
DIAGNOSIS: control
MMSE_SCORE: 30
REASONING: Highly detailed, organized description with rich vocabulary and complex sentence structures. Patient demonstrates excellent observational skills, narrative coherence, and ability to make inferences (mother is "distracted," creating "sense of danger"). No evidence of cognitive impairment.

---
"""

# Load model and processor - FULL MODEL WITHOUT QUANTIZATION
print("\nLoading FULL model without quantization...")
torch.cuda.empty_cache()

# Load processor
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

# Load the full model without quantization
try:
    print("Loading full model on GPU (no quantization)...")
    model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct",
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    print("✅ Full model loaded successfully on GPU!")
    
except Exception as e:
    print(f"❌ GPU loading failed: {str(e)}")
    print("🔄 Falling back to CPU...")
    
    torch.cuda.empty_cache()
    
    model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct",
        torch_dtype=torch.float16,
        device_map="cpu",
        low_cpu_mem_usage=True
    )
    print("✅ Model loaded successfully on CPU! (This will be slower)")

print("Model loading complete! Using FULL MODEL without quantization.\n")

# Store results
results = []

# SANITY CHECK: Sample a few transcriptions first
print("\n" + "="*80)
print("SANITY CHECK: Sampling first 3 transcriptions...")
print("="*80)
for i, cha_file in enumerate(cha_files[:3]):
    file_name = os.path.basename(cha_file)
    print(f"\nFile {i+1}: {file_name}")
    with open(cha_file, 'r', encoding='utf-8', errors='replace') as f:
        sample = f.read().strip()
    print(f"Length: {len(sample)} chars, {len(sample.split())} words")
    print(f"Content preview:\n{sample[:400]}...")
    print("-" * 80)

print("\n" + "="*80)
print("Starting full processing...")
print("="*80 + "\n")

# Process each file
for cha_file in tqdm(cha_files, desc="Processing files"):
    file_name = os.path.basename(cha_file)
    file_id = os.path.splitext(file_name)[0]
    
    print(f"\n{'='*80}")
    print(f"Processing: {file_name} (ID: {file_id})")
    print('='*80)
    
    try:
        with open(cha_file, 'r', encoding='utf-8', errors='replace') as f:
            transcription = f.read().strip()
        
        if len(transcription.strip()) < 10:
            print(f"⚠️ Empty/short transcription - marking as 'error'")
            results.append({
                'id': file_id,
                'file': file_name,
                'predicted_diagnosis': 'error',
                'predicted_mmse': -1,
                'reasoning': 'Transcription too short or empty',
                'transcription_length': len(transcription)
            })
            continue
        
        print(f"Transcription preview: {transcription[:200]}...")
        
        # FEW-SHOT PROMPT with examples
        messages_diagnosis = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"""You are a clinical expert in dementia assessment. Analyze Cookie Theft picture descriptions to assess cognitive status.

I will show you several examples of how to assess transcriptions, then you will assess a new patient.

{FEW_SHOT_EXAMPLES}

Now, analyze this NEW PATIENT'S transcription using the same assessment approach:

PATIENT'S TRANSCRIPTION:
{transcription}

Based on the coherence, content, and linguistic quality compared to the examples above, provide your clinical assessment in this EXACT format:

DIAGNOSIS: [control/probablead]
MMSE_SCORE: [0-30]
REASONING: [2-3 sentence explanation]

Analyze THIS patient now:"""}
                ]
            }
        ]
        
        text_prompt_diagnosis = processor.apply_chat_template(messages_diagnosis, add_generation_prompt=True, tokenize=False)
        inputs_diagnosis = processor(text=[text_prompt_diagnosis], images=[image], padding=True, return_tensors="pt")
        
        inputs_diagnosis = {k: v.to(model.device) for k, v in inputs_diagnosis.items()}
        
        input_length = inputs_diagnosis["input_ids"].shape[-1]
        
        with torch.no_grad():
            outputs_diagnosis = model.generate(
                **inputs_diagnosis, 
                max_new_tokens=400,
                do_sample=True,
                temperature=1.0,  # Higher temperature for more diversity
                top_p=0.92,       # Slightly lower top_p for more varied sampling
                top_k=50,         # Add top_k sampling for diversity
                repetition_penalty=1.1,  # Discourage repetition
                pad_token_id=processor.tokenizer.eos_token_id
            )
        
        del inputs_diagnosis
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        diagnosis_result = processor.decode(outputs_diagnosis[0][input_length:], skip_special_tokens=True)
        
        # === DEBUGGING: PRINT RAW OUTPUT AND TRANSCRIPTION ===
        print("\n" + "="*60)
        # ENHANCED EXTRACTION with multiple fallback strategies
        diagnosis = None
        
        # Strategy 1: Look for "DIAGNOSIS:" followed by the label
        diagnosis_match = re.search(r'DIAGNOSIS:\s*(\bprobablead\b|\bcontrol\b)', diagnosis_result, re.IGNORECASE)
        if diagnosis_match:
            diagnosis = diagnosis_match.group(1).lower()
        
        # Strategy 2: Look anywhere in first 200 chars for the labels
        if not diagnosis:
            early_match = re.search(r'(\bprobablead\b|\bcontrol\b)', diagnosis_result[:200], re.IGNORECASE)
            if early_match:
                diagnosis = early_match.group(1).lower()
        
        # Strategy 3: Look for common variations like "probable AD", "probable alzheimer"
        if not diagnosis:
            if re.search(r'\bprobable\s*a\.?d\.?\b|\balzheimer', diagnosis_result, re.IGNORECASE):
                diagnosis = 'probablead'
            elif re.search(r'\bhealthy\b|\bnormal\b|\bno\s+dementia\b', diagnosis_result, re.IGNORECASE):
                diagnosis = 'control'
        
        # Strategy 4: Search entire text as last resort
        if not diagnosis:
            full_match = re.search(r'(\bprobablead\b|\bcontrol\b)', diagnosis_result, re.IGNORECASE)
            if full_match:
                diagnosis = full_match.group(1).lower()
        
        reasoning_match = re.search(r'REASONING:.*?(.*)', diagnosis_result, re.IGNORECASE | re.DOTALL)
        
        # MMSE SCORE extraction with multiple fallback strategies
        mmse_score = -1  # Default to -1 (unknown)
        mmse_start_keyword = 'mmse_score'
        
        mmse_start_index = diagnosis_result.lower().find(mmse_start_keyword)

        if mmse_start_index != -1:
            search_segment = diagnosis_result[mmse_start_index + len(mmse_start_keyword):]
            mmse_match = re.search(r'\d+', search_segment)
            
            if mmse_match:
                try:
                    candidate_score = int(mmse_match.group(0))
                    # Validate MMSE score is in valid range (0-30)
                    if 0 <= candidate_score <= 30:
                        mmse_score = candidate_score
                    else:
                        print(f"⚠️ Invalid MMSE score {candidate_score}, setting to -1 (unknown)")
                except ValueError:
                    pass
        
        # If still no MMSE found, try broader search
        if mmse_score == -1:
            # Look for any number between 0-30 after keywords like "score", "mmse", "estimate"
            mmse_pattern = re.search(r'(?:score|mmse|estimate).*?(\d+)', diagnosis_result, re.IGNORECASE)
            if mmse_pattern:
                try:
                    candidate_score = int(mmse_pattern.group(1))
                    if 0 <= candidate_score <= 30:
                        mmse_score = candidate_score
                except ValueError:
                    pass
        
        # Final check - if still no diagnosis, mark as 'unknown'
        if not diagnosis:
            diagnosis = 'unknown'
            print(f"⚠️ No diagnosis found in model output, marked as 'unknown'")
            print(f"   Model output was: {diagnosis_result[:300]}...")
        
        reasoning = reasoning_match.group(1).strip() if reasoning_match else diagnosis_result[:200]
        
        # Print debugging output
        print("TRANSCRIPTION ANALYSIS:")
        print(f"  Length: {len(transcription)} chars, {len(transcription.split())} words")
        print(f"  Preview: {transcription[:300]}...")
        print("\nMODEL OUTPUT:")
        print(diagnosis_result)
        print("="*60 + "\n")
        
        results.append({
            'id': file_id,
            'file': file_name,
            'predicted_diagnosis': diagnosis,
            'predicted_mmse': mmse_score,
            'reasoning': reasoning,
            'transcription_length': len(transcription)
        })
        
        print(f"✅ Processed: {diagnosis} (MMSE: {mmse_score if mmse_score != -1 else 'unknown'})")
        print(f"   Transcription stats: {len(transcription)} chars, {len(transcription.split())} words")
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"❌ ERROR processing {file_name}: {str(e)}")
        print(f"Marking as 'error' with MMSE=-1")
        import traceback
        traceback.print_exc()
        
        # Add error entry instead of skipping
        results.append({
            'id': file_id,
            'file': file_name,
            'predicted_diagnosis': 'error',
            'predicted_mmse': -1,
            'reasoning': f'Processing error: {str(e)[:100]}',
            'transcription_length': 0
        })
        continue

print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE!")
print(f"Total files processed: {len(results)}")
print('='*80)

# Create DataFrame
df_results = pd.DataFrame(results)

# Count error/unknown predictions
error_count = (df_results['predicted_diagnosis'] == 'error').sum()
unknown_count = (df_results['predicted_diagnosis'] == 'unknown').sum()
valid_count = len(df_results) - error_count - unknown_count

print(f"\nResults breakdown:")
print(f"  Valid predictions: {valid_count}")
print(f"  Unknown predictions: {unknown_count}")
print(f"  Error predictions: {error_count}")
print(f"  Total: {len(df_results)}")

print("\nResults summary:")
print(df_results.head())

# Print statistics (excluding error/unknown)
valid_results = df_results[~df_results['predicted_diagnosis'].isin(['error', 'unknown'])]
print(f"\nDiagnosis distribution (valid predictions only):")
if len(valid_results) > 0:
    diagnosis_counts = valid_results['predicted_diagnosis'].value_counts()
    print(diagnosis_counts)
    print(f"\nPercentages:")
    print(diagnosis_counts / len(valid_results) * 100)
else:
    print("  No valid predictions")

print(f"\nMMSE score statistics (excluding -1/unknown):")
valid_mmse = df_results[df_results['predicted_mmse'] >= 0]['predicted_mmse']
if len(valid_mmse) > 0:
    print(f"  Mean: {valid_mmse.mean():.2f}")
    print(f"  Std:  {valid_mmse.std():.2f}")
    print(f"  Min:  {valid_mmse.min()}")
    print(f"  Max:  {valid_mmse.max()}")
    print(f"  Median: {valid_mmse.median():.2f}")
else:
    print("  No valid MMSE scores found")

# ============================================================================
# MERGE WITH GROUND TRUTH AND CALCULATE METRICS
# ============================================================================
print(f"\n{'='*80}")
print("CALCULATING METRICS")
print('='*80)

# Merge predictions with ground truth (use df_gt_valid which only has records with dx)
df_merged = df_results.merge(df_gt_valid, on='id', how='inner')
print(f"\nMerged {len(df_merged)} records with ground truth")

# All merged records have valid dx (since we used df_gt_valid)
if len(df_merged) > 0:
    # For accuracy calculation, error/unknown count as wrong
    y_true = df_merged['dx'].values
    y_pred = df_merged['predicted_diagnosis'].values
    
    print(f"\nUnique true labels: {np.unique(y_true)}")
    print(f"Unique predicted labels: {np.unique(y_pred)}")
    
    # Count predictions
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(df_merged)
    accuracy_all = correct_predictions / total_predictions
    
    error_unknown_count = df_merged['predicted_diagnosis'].isin(['error', 'unknown']).sum()
    
    print(f"\n--- CLASSIFICATION METRICS (error/unknown count as WRONG) ---")
    print(f"Total with ground truth: {total_predictions}")
    print(f"Correct predictions: {correct_predictions}")
    print(f"Wrong predictions: {total_predictions - correct_predictions}")
    print(f"  - of which error/unknown: {error_unknown_count}")
    print(f"Accuracy (ALL): {accuracy_all:.4f}")
    
    # Calculate standard metrics only on valid predictions (excluding error/unknown)
    df_valid = df_merged[~df_merged['predicted_diagnosis'].isin(['error', 'unknown'])].copy()
    
    if len(df_valid) > 0:
        y_true_valid = df_valid['dx'].values
        y_pred_valid = df_valid['predicted_diagnosis'].values
        
        accuracy_valid = accuracy_score(y_true_valid, y_pred_valid)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true_valid, y_pred_valid, 
            average='binary', 
            pos_label='probablead',
            zero_division=0
        )
        
        # Confusion Matrix (valid predictions only)
        cm = confusion_matrix(y_true_valid, y_pred_valid, labels=['probablead', 'control'])
        tp, fn, fp, tn = cm.ravel()
        
        # Specificity and Sensitivity
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        print(f"\n--- METRICS ON VALID PREDICTIONS ONLY (excluding error/unknown) ---")
        print(f"Valid predictions evaluated: {len(df_valid)}")
        print(f"Accuracy:    {accuracy_valid:.4f}")
        print(f"Precision:   {precision:.4f}")
        print(f"Recall:      {recall:.4f}")
        print(f"F1-Score:    {f1:.4f}")
        print(f"Sensitivity: {sensitivity:.4f}")
        print(f"Specificity: {specificity:.4f}")
        
        print("\nConfusion Matrix (valid predictions only):")
        print(f"                  Predicted")
        print(f"Actual       probableAD  control")
        print(f"probableAD     {tp:5d}    {fn:5d}")
        print(f"control        {fp:5d}    {tn:5d}")
    else:
        print("\n⚠️ No valid predictions to calculate detailed metrics")
        accuracy_valid = 0
        precision = recall = f1 = sensitivity = specificity = 0
        tp = tn = fp = fn = 0
    
    # MMSE Metrics (only for records with ground truth mmse, -1 counts as wrong)
    records_with_mmse = df_merged[df_merged['mmse'].notna()].copy()
    
    if len(records_with_mmse) > 0:
        # Count how many MMSE predictions match (excluding -1)
        correct_mmse = (records_with_mmse['mmse'] == records_with_mmse['predicted_mmse']).sum()
        total_mmse = len(records_with_mmse)
        error_mmse = (records_with_mmse['predicted_mmse'] == -1).sum()
        
        print(f"\n--- MMSE METRICS (error/-1 counts as WRONG) ---")
        print(f"Total with ground truth MMSE: {total_mmse}")
        print(f"Exact matches: {correct_mmse}")
        print(f"Wrong predictions: {total_mmse - correct_mmse}")
        print(f"  - of which error/unknown (-1): {error_mmse}")
        
        # For MAE/RMSE, only use valid predictions (not -1)
        df_valid_mmse = records_with_mmse[records_with_mmse['predicted_mmse'] >= 0]
        
        if len(df_valid_mmse) > 0:
            y_true_mmse = df_valid_mmse['mmse'].values
            y_pred_mmse = df_valid_mmse['predicted_mmse'].values
            
            mae = mean_absolute_error(y_true_mmse, y_pred_mmse)
            mse = mean_squared_error(y_true_mmse, y_pred_mmse)
            rmse = np.sqrt(mse)
            
            print(f"\nRegression metrics (valid predictions only, excluding -1):")
            print(f"MAE (Mean Absolute Error): {mae:.4f}")
            print(f"MSE (Mean Squared Error):  {mse:.4f}")
            print(f"RMSE (Root MSE):           {rmse:.4f}")
            print(f"Samples evaluated:         {len(df_valid_mmse)}")
        else:
            print("\nNo valid MMSE predictions (all were -1/error)")
            mae = mse = rmse = np.nan
    else:
        print("\n--- MMSE METRICS ---")
        print("No records with ground truth MMSE")
        mae = mse = rmse = np.nan
        correct_mmse = total_mmse = error_mmse = 0
    
    # Create metrics summary rows
    metrics_summary = [
        {'id': '', 'file': '--- METRICS SUMMARY ---', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Accuracy (ALL, error=wrong)', 'predicted_diagnosis': '', 'predicted_mmse': accuracy_all, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Accuracy (valid only)', 'predicted_diagnosis': '', 'predicted_mmse': accuracy_valid, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Precision', 'predicted_diagnosis': '', 'predicted_mmse': precision, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Recall', 'predicted_diagnosis': '', 'predicted_mmse': recall, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'F1-Score', 'predicted_diagnosis': '', 'predicted_mmse': f1, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Sensitivity', 'predicted_diagnosis': '', 'predicted_mmse': sensitivity, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'METRIC', 'file': 'Specificity', 'predicted_diagnosis': '', 'predicted_mmse': specificity, 'reasoning': '', 'transcription_length': np.nan},
        {'id': '', 'file': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'CONFUSION', 'file': 'True Positives', 'predicted_diagnosis': '', 'predicted_mmse': int(tp), 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'CONFUSION', 'file': 'True Negatives', 'predicted_diagnosis': '', 'predicted_mmse': int(tn), 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'CONFUSION', 'file': 'False Positives', 'predicted_diagnosis': '', 'predicted_mmse': int(fp), 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'CONFUSION', 'file': 'False Negatives', 'predicted_diagnosis': '', 'predicted_mmse': int(fn), 'reasoning': '', 'transcription_length': np.nan},
        {'id': '', 'file': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'MMSE', 'file': 'MAE', 'predicted_diagnosis': '', 'predicted_mmse': mae, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'MMSE', 'file': 'MSE', 'predicted_diagnosis': '', 'predicted_mmse': mse, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'MMSE', 'file': 'RMSE', 'predicted_diagnosis': '', 'predicted_mmse': rmse, 'reasoning': '', 'transcription_length': np.nan},
        {'id': '', 'file': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Total Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_results), 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Valid DX Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_valid) if len(df_valid) > 0 else 0, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Error/Unknown DX', 'predicted_diagnosis': '', 'predicted_mmse': error_unknown_count, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Valid MMSE Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_valid_mmse) if len(records_with_mmse) > 0 and len(df_valid_mmse) > 0 else 0, 'reasoning': '', 'transcription_length': np.nan},
        {'id': 'COUNT', 'file': 'Error/Unknown MMSE (-1)', 'predicted_diagnosis': '', 'predicted_mmse': error_mmse, 'reasoning': '', 'transcription_length': np.nan},
    ]
    
    df_with_metrics = pd.concat([df_results, pd.DataFrame(metrics_summary)], ignore_index=True)
    
else:
    print("\n⚠️ No records with ground truth to evaluate!")
    df_with_metrics = df_results

# ============================================================================
# SAVE RESULTS TO CSV
# ============================================================================
try:
    df_with_metrics.to_csv(output_csv_path, index=False)
    print(f"\n{'='*80}")
    print(f"✅ PREDICTIONS AND METRICS SAVED TO CSV!")
    print(f"Output file: {output_csv_path}")
    print(f"Total records saved: {len(df_with_metrics)}")
    print(f"(Includes {len(df_results)} predictions + metrics summary)")
    print(f"\nCSV includes:")
    print(f"  - Valid predictions: {valid_count}")
    print(f"  - Unknown predictions (pred_dx='unknown', pred_mmse=-1): {unknown_count}")
    print(f"  - Error predictions (pred_dx='error', pred_mmse=-1): {error_count}")
    print(f"\nNote: {skipped_count} files were skipped (no dx label in ground truth)")
    print('='*80)
    
except Exception as e:
    print(f"\n❌ ERROR saving CSV: {str(e)}")
    print("Results are still available in the df_results DataFrame")

print("\n" + "="*80)
print("Script execution completed!")
print("="*80)